# 02 · Baseline: standard GPT, 50M tokens, batch 32

The ordinary residual network, stored activations, plain autograd. Batch 32 x 512 = 16,384 tokens a step,
3,051 steps. Batch 32 is the fixed batch for the first two runs: it is the largest power of two the
**standard** model trains at on this machine (notebook 04 finds the exact ceiling).

In [1]:
import sys, os, json, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not pathlib.Path("ERA_V5").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amukka/ERA_V5.git"], check=True)
    os.chdir("ERA_V5/session13_Distributed_Training")
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
if not (ROOT / "data" / "train.bin").exists():          # ~5 minutes: stream FineWeb-Edu, train the BPE
    subprocess.run([sys.executable, "tools/prepare_data.py"], check=True)

import torch, platform
from src import experiments as X
from src.train import pick_device
dev = pick_device()
print("device:", dev, "| torch", torch.__version__, "| python", platform.python_version(), "|", platform.platform())
if dev == "mps":
    print(f"MPS recommended working set: {torch.mps.recommended_max_memory()/2**30:.2f} GiB")
if dev == "cuda":
    print(torch.cuda.get_device_name(), f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
print(json.dumps(json.loads((ROOT / "data" / "meta.json").read_text()), indent=1))

device: mps | torch 2.13.0 | python 3.14.5 | macOS-26.6.2-arm64-arm-64bit-Mach-O
MPS recommended working set: 11.84 GiB
{
 "source": "HuggingFaceFW/fineweb-edu, config sample-10BT, streamed in order",
 "vocab_size": 8192,
 "eos_id": 0,
 "train_tokens": 56000000,
 "train_docs": 46080,
 "val_tokens": 1000000,
 "val_docs": 1024,
 "chars_per_token": 3.886,
 "note": "val.bin comes from the tokenizer's training documents; train.bin from the documents after them",
 "seconds": 314.3
}


**The model** (identical in every run): GPT, 10 layers, d_model 384, 6 heads, MLP 4x, sequence 512,
byte-level BPE vocabulary 8,192 with the head tied to the embedding. **21,053,184 parameters**, of which
17,710,848 are outside the embedding tables. No dropout anywhere, because a reversible backward pass
recomputes each block and a random mask would recompute differently. AdamW (0.9, 0.95, wd 0.1), peak LR
1e-3, 3% warmup then cosine to 10%, gradient clip 1.0, fp32.

In [2]:
base = X.train(name="baseline_standard_b32", variant="standard", batch=X.BASE_BATCH, tokens=X.TOKENS,
               eval_every=250, log_every=10)

{"name": "baseline_standard_b32", "device": "mps", "config": {"vocab_size": 8192, "seq_len": 512, "n_layer": 10, "n_head": 6, "d_model": 384, "variant": "standard", "h": 1.0, "fp_iters": 6, "extra": {}}, "params": 21053184, "params_non_embedding": 17710848, "batch": 32, "tokens_per_step": 16384, "steps": 3051, "peak_lr": 0.001, "warmup": 91, "amp_bf16": 0, "torch": "2.13.0", "machine": "arm64", "processor": "arm"}


step     0/3051  tok   0.02M  loss 9.0926  gnorm 6.631  0 tok/s  0.0 min


memory audit at step 3: exact peak 9,372 MiB in aten._log_softmax_backward_data.default


step   100/3051  tok   1.64M  loss 6.3083  gnorm 0.444  11,028 tok/s  2.5 min


step   200/3051  tok   3.28M  loss 5.9581  gnorm 0.400  11,106 tok/s  5.0 min


...

            eval @ 49.14M tokens: val loss 3.9847


step  3000/3051  tok  49.15M  loss 4.0283  gnorm 0.535  11,168 tok/s  74.6 min


step  3050/3051  tok  49.97M  loss 3.9708  gnorm 0.529  10,965 tok/s  75.8 min


            eval @ 49.97M tokens: val loss 3.9801


{"tok_per_s": 10982.8, "final_train_loss": 3.9707937240600586, "final_val_loss": 3.980102550983429, "peak_exact_mib": 9371.5, "peak_alloc_mib": 8706.5, "peak_driver_mib": 11835.0, "train_seconds": 4550.0}


In [3]:
keys = ["tokens_trained", "steps", "final_train_loss", "final_train_loss_avg_last5pct", "final_val_loss",
        "tok_per_s", "train_seconds", "peak_exact_mib", "peak_exact_op", "peak_alloc_mib", "peak_driver_mib"]
print(json.dumps({k: base[k] for k in keys}, indent=1))

{
 "tokens_trained": 49971200,
 "steps": 3051,
 "final_train_loss": 3.9707937240600586,
 "final_train_loss_avg_last5pct": 3.9758702754974364,
 "final_val_loss": 3.980102550983429,
 "tok_per_s": 10982.8,
 "train_seconds": 4550.0,
 "peak_exact_mib": 9371.5,
 "peak_exact_op": "aten._log_softmax_backward_data.default",
 "peak_alloc_mib": 8706.5,
 "peak_driver_mib": 11835.0
}
